#PyTorch Training Pipeline

Base model: MARBERTv2

**Setup**

In [ ]:
import os
from google.colab import userdata

!git config --global user.name "sivrox"
repo_url = "https://github.com/sivrox/Arabic-English-Sentiment-Analysis-Project.git"

if not os.path.exists("/content/Arabic-English-Sentiment-Analysis-Project"):
    os.system(f"git clone {repo_url}")
else:
    os.system("cd /content/Arabic-English-Sentiment-Analysis-Project && git pull")

%cd /content/Arabic-English-Sentiment-Analysis-Project
print("Repository cloned.")
!ls

/content/Arabic-English-Sentiment-Analysis-Project
Repository cloned.
configs     models		    preprocessing	    session.tw_session
deployment  notebooks		    README.md
evaluation  peft_implementation.py  results_comparison.csv


In [ ]:
#install required packages
!pip install -q transformers==4.39.0 peft==0.10.0 accelerate==0.28.0 scikit-learn evaluate seaborn matplotlib nltk pyarrow pandas datasets tqdm emoji

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 124.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.39.0 which is incompatible.


In [ ]:
import os, sys, random, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from tqdm import tqdm
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import evaluate
import nltk
nltk.download("punkt", quiet=True)

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

#add repo root to path so we can import our preprocessing and peft modules
sys.path.insert(0, "/content/Arabic-English-Sentiment-Analysis-Project")

#hyperparameters
model_name = "UBC-NLP/MARBERTv2"
max_len = 128
batch_size = 16
epochs = 3
lr_full = 2e-5
lr_lora = 3e-4
num_labels = 3
patience = 3

#label encoding
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {"negative": 0, "neutral": 1, "positive": 2}

#output folder for plots and CSVs
Path("results/pytorch").mkdir(parents=True, exist_ok=True)
print("Config ready.")

Device: cuda
GPU  : NVIDIA A100-SXM4-40GB
VRAM : 42.4 GB
Config ready.


**Load Data**

In [ ]:
from preprocessing.preprocessor import build_dataloaders

data_path = "preprocessing/datasets/processed/unified_raw.csv"
print("Building DataLoaders for MARBERT...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_loader, val_loader, test_loader = build_dataloaders(data_path, tokenizer, batch_size=batch_size, max_length=max_len, seed=seed, save_cleaned_csv=True)

#verifying batch shapes and label encoding before training
sample = next(iter(train_loader))
print(f"\nBatch check:")
print(f"input_ids shape : {sample['input_ids'].shape}")
print(f"labels present : {sample['label'].unique().tolist()} (0=negative, 1=neutral, 2=positive)")

Building DataLoaders for MARBERT...


tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


Preprocessing pipeline...
Input: preprocessing/datasets/processed/unified_raw.csv
Tokenizer: UBC-NLP/MARBERTv2
Loaded: 306,880 rows

Cleaning and normalizing text...
After filtering: 301,588 rows (removed 5,292)

Split distribution:
  train: 237,033
  test: 32,340
  val: 29,629
  gold_test: 2,586
Cleaned dataset saved: preprocessing/datasets/processed/cleaned_dataset.csv
Training set after balancing: 151,242
Label counts: {'positive': np.int64(60886), 'neutral': np.int64(45178), 'negative': np.int64(45178)}

Tokenizing...
DataLoaders ready - Train: 9,453 | Val: 1,852 | Test: 2,022 batches

Batch check:
input_ids shape : torch.Size([16, 128])
labels present : [0, 1, 2] (0=negative, 1=neutral, 2=positive)


**Helper Functions**

In [ ]:
f1_metric = evaluate.load("f1")

def train_epoch(model, loader, optimizer, scheduler, epoch, total_epochs): #loops over the training set and update the model weights
    model.train()
    total_loss, all_preds, all_labels = 0, [], []
    progress = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs}", leave=True)
    for batch in progress:
        optimizer.zero_grad()
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labs = batch["label"].to(device)
        # forward pass — model returns loss directly when labels are provided
        out  = model(input_ids=ids, attention_mask=mask, labels=labs)
        out.loss.backward()
        # clip gradients to prevent exploding gradients during fine-tuning
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += out.loss.item()
        all_preds.extend(out.logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(labs.cpu().tolist())
        # show running batch loss on the progress bar
        progress.set_postfix(loss=f"{out.loss.item():.4f}")
    avg_loss = total_loss / len(loader)
    f1 = f1_metric.compute(predictions=all_preds, references=all_labels, average="macro")["f1"]
    return avg_loss, f1

@torch.no_grad()
def evaluate_model(model, loader): #evaluation loop (does not update the weights)
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []
    for batch in loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labs = batch["label"].to(device)
        out  = model(input_ids=ids, attention_mask=mask, labels=labs)
        total_loss += out.loss.item()
        all_preds.extend(out.logits.argmax(dim=-1).cpu().tolist())
        all_labels.extend(labs.cpu().tolist())
    avg_loss = total_loss / len(loader)
    f1  = f1_metric.compute(predictions=all_preds, references=all_labels, average="macro")["f1"]
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, f1, all_preds, all_labels, acc

def plot_curves(history, title, save_path): #to plot f1 and losses curves
    ep = range(1, len(history["train_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(ep, history["train_loss"], label="Train")
    ax1.plot(ep, history["val_loss"],   label="Val")
    ax1.set(title=f"{title} — Loss", xlabel="Epoch", ylabel="Loss")
    ax1.legend()
    ax2.plot(ep, history["train_f1"], label="Train")
    ax2.plot(ep, history["val_f1"],   label="Val")
    ax2.set(title=f"{title} — Macro F1", xlabel="Epoch", ylabel="F1")
    ax2.legend()
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

def plot_confusion_matrix(preds, labels, title, save_path): #to plot per-class prediction breakdown
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Negative", "Neutral", "Positive"],
                yticklabels=["Negative", "Neutral", "Positive"])
    plt.title(f"{title} — Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    plt.close()

def print_report(preds, labels, title):
    print(f"\n{title} — Classification Report")
    print(classification_report(labels, preds, target_names=["Negative", "Neutral", "Positive"]))

def run_training(model, n_epochs, lr): #training loop for both Full FT and LoRA strategies
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=lr, weight_decay=0.01)

    total_steps = len(train_loader) * n_epochs
    scheduler   = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    best_val_f1, best_state, no_improve = 0, None, 0

    for epoch in range(1, n_epochs + 1):
        tr_loss, tr_f1 = train_epoch(model, train_loader, optimizer, scheduler, epoch, n_epochs)
        vl_loss, vl_f1, _, _, vl_acc = evaluate_model(model, val_loader)

        history["train_loss"].append(tr_loss)
        history["val_loss"].append(vl_loss)
        history["train_f1"].append(tr_f1)
        history["val_f1"].append(vl_f1)

        #print epoch status
        print(f"val_loss={vl_loss:.4f}  val_f1={vl_f1:.4f}  accuracy={vl_acc:.4f}")

        if vl_f1 > best_val_f1:
            best_val_f1 = vl_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}  #stores best weights in RAM to avoid repeated disk writes during training
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    #restore best checkpoint before returning
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
    return model, history, best_val_f1

Strategy A - **Full Fine-Tuning**

In [6]:
#load pretrained MARBERT with a new 3-class classification head
print("Loading MARBERT for Full Fine-Tuning...")
model_fft = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True).to(device)
total_params = sum(p.numel() for p in model_fft.parameters())
trainable_fft = sum(p.numel() for p in model_fft.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {trainable_fft:,} / {total_params:,} ({100*trainable_fft/total_params:.1f}%)")

print("\nTraining MARBERT Full Fine-Tuning...")
model_fft, history_fft, best_val_fft = run_training(model_fft, epochs, lr_full)

#save in HuggingFace folder format for deployment
model_fft.save_pretrained("best_marbert_fft")
tokenizer.save_pretrained("best_marbert_fft")
print(f"\nBest val F1: {best_val_fft:.4f}")

Loading MARBERT for Full Fine-Tuning...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at UBC-NLP/MARBERTv2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Trainable parameters: 162,843,651 / 162,843,651 (100.0%)

Training MARBERT Full Fine-Tuning...


...

val_loss=0.2864  val_f1=0.8794  accuracy=0.8992


Epoch 2/5: 100%|██████████| 9453/9453 [14:19<00:00, 11.00it/s, loss=0.0364]


val_loss=0.3025  val_f1=0.8879  accuracy=0.9067


Epoch 3/5:   2%|▏         | 193/9453 [00:17<14:03, 10.97it/s, loss=0.0075]


KeyboardInterrupt: 

In [ ]:
#plot training curves
plot_curves(history_fft, "MARBERT Full FT", "results/pytorch/marbert_fft_curves.png")
_, test_f1_fft, preds_fft, labels_test, acc_fft = evaluate_model(model_fft, test_loader)
print(f"MARBERT Full FT — Test Macro F1: {test_f1_fft:.4f}")
print_report(preds_fft, labels_test, "MARBERT Full FT")

#plot confusion matrix
plot_confusion_matrix(preds_fft, labels_test, "MARBERT Full FT", "results/pytorch/marbert_fft_cm.png")

Strategy B  **LoRA from Scratch**

In [ ]:
from peft_implementation import inject_lora, count_parameters

print("Loading MARBERT for LoRA from Scratch...")
lora_base  = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels, id2label=id2label, label2id=label2id,ignore_mismatched_sizes=True)
model_lora = inject_lora(lora_base, target_modules=("query", "value"), r=8, alpha=16.0)

for name, param in model_lora.named_parameters():
    if "classifier" in name:
        param.requires_grad = True

model_lora  = model_lora.to(device)
params_lora = count_parameters(model_lora)
print(f"\nParameter reduction: {trainable_fft:,} → {params_lora['trainable']:,} trainable params")
print(f"\nReduction factor : {trainable_fft / params_lora['trainable']:.0f}x fewer trainable parameters")

print("\nTraining MARBERT LoRA from Scratch...")
model_lora, history_lora, best_val_lora = run_training(model_lora, epochs, lr_lora)

#save as .pt state dict
torch.save(model_lora.state_dict(), "best_marbert_lora.pt")
print(f"\nBest val F1: {best_val_lora:.4f}")

In [ ]:
#plot training curves
plot_curves(history_lora, "MARBERT LoRA", "results/pytorch/marbert_lora_curves.png")
_, test_f1_lora, preds_lora, _, acc_lora = evaluate_model(model_lora, test_loader)
print(f"MARBERT LoRA — Test Macro F1: {test_f1_lora:.4f}")
print_report(preds_lora, labels_test, "MARBERT LoRA")

#plot confusion matrix
plot_confusion_matrix(preds_lora, labels_test, "MARBERT LoRA", "results/pytorch/marbert_lora_cm.png")

**Results Comparison**

In [ ]:
#convert test metric to dataframe
results_df = pd.DataFrame({
    "model" : ["MARBERT FFT", "MARBERT LoRA"],
    "pipeline" : ["PyTorch", "PyTorch"],
    "test_macro_f1" : [round(test_f1_fft, 4), round(test_f1_lora, 4)],
    "test_accuracy" : [round(acc_fft, 4), round(acc_lora, 4)],
    "trainable_params": [trainable_fft, params_lora["trainable"]],
    "best_val_f1" : [round(best_val_fft, 4), round(best_val_lora, 4)],})

print("PyTorch Results — MARBERT:")
print(results_df.to_string(index=False))

#save dataframe as CSV
results_df.to_csv("results/pytorch/pytorch_comparison.csv", index=False)
print("\nSaved: results/pytorch/pytorch_comparison.csv")

In [ ]:
#cross-strategy comparison
fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(results_df))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], results_df["test_macro_f1"], width, label="Macro F1", color="#4C72B0", alpha=0.85)
bars2 = ax.bar([i + width/2 for i in x], results_df["test_accuracy"], width, label="Accuracy", color="#55A868", alpha=0.85)
ax.bar_label(bars1, fmt="%.4f", padding=3, fontsize=9)
ax.bar_label(bars2, fmt="%.4f", padding=3, fontsize=9)
ax.set_xticks(list(x))
ax.set_xticklabels([r.strategy for r in results_df.itertuples()])
ax.set_ylim(0.7, 1.0)
ax.set_ylabel("Score")
ax.set_title("PyTorch Results — MARBERT Full FT vs LoRA")
ax.legend()
plt.tight_layout()
plt.savefig("results/pytorch/pytorch_comparison_chart.png", dpi=150)
plt.show()
plt.close()

**Save to GitHub**

In [ ]:
#ignore model files
with open(".gitignore", "w") as f:
    f.write("*.pt\n*.bin\nbest_*\ndrive/\n__pycache__/\n*.pyc\n")

#push results (CSVs and PNGs)
os.system("git add results/ preprocessing/datasets/processed/cleaned_dataset.csv .gitignore -f")
os.system('git commit -m "Add PyTorch training results"')
os.system("git push origin main")
print("Pushed results to GitHub.")

**Save trained models in Google Drive**

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
save_dir = "/content/drive/MyDrive/arabic-english-sentiment-analysis-models"
os.makedirs(save_dir, exist_ok=True)

#Full FT checkpoint
if os.path.exists("best_marbert_fft"):
    dest = os.path.join(save_dir, "best_marbert_fft")
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree("best_marbert_fft", dest)
    print(f"Saved: best_marbert_fft → {dest}")

#LoRA checkpoint
if os.path.exists("best_marbert_lora.pt"):
    shutil.copy("best_marbert_lora.pt", os.path.join(save_dir, "best_marbert_lora.pt"))
    print(f"Saved: best_marbert_lora.pt → {save_dir}")

print(f"\nAll checkpoints saved to Google Drive: {save_dir}")
print("Note: best_marbert_fft is used for Docker deployment.")
print("best_marbert_lora.pt requires the model architecture to reload.")